<a href="https://colab.research.google.com/github/hydroz3/flyrank-ml-intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hydroz3/flyrank-ml-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')
print("Token loaded:", token is not None)

Token loaded: True


In [5]:
from huggingface_hub import whoami

whoami(token=token)

{'type': 'user',
 'id': '6a8151b19a93e0766c4b1be8',
 'name': 'hydroz',
 'fullname': 'hydroz',
 'email': 'richard.141206@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1788220800,
 'isPro': False,
 'avatarUrl': '/avatars/c1fef3d0faeec2b0fa1143ec9fd5b3ab.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'flyrank-read-token',
   'role': 'read',
   'createdAt': '2026-08-17T14:48:10.124Z'}}}

In [6]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=token
)

files[:10]

['.gitattributes',
 'README.md',
 'dim_clients.parquet',
 'dim_content.parquet',
 'fact_content_daily_performance/month=2025-01/data_0.parquet',
 'fact_content_daily_performance/month=2025-02/data_0.parquet',
 'fact_content_daily_performance/month=2025-03/data_0.parquet',
 'fact_content_daily_performance/month=2025-04/data_0.parquet',
 'fact_content_daily_performance/month=2025-05/data_0.parquet',
 'fact_content_daily_performance/month=2025-06/data_0.parquet']

In [19]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [20]:
import duckdb
con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")
# accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

In [9]:
print(rel)

hf://datasets/FlyRank/internship-warehouse


In [10]:
con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [29]:
import duckdb

# Connect to DuckDB
con = duckdb.connect()

# Authenticate DuckDB with your Hugging Face token
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

# FlyRank warehouse root
rel = "hf://datasets/FlyRank/internship-warehouse"

# Preview 10 rows from March 2026
result = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet'
    )
    LIMIT 10
""").df()

result


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,True,False,True,<NA>,239,1,1756,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,True,False,True,<NA>,191,0,1496,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,True,False,True,<NA>,55,0,180,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,True,False,True,<NA>,77,0,434,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,True,False,True,<NA>,2,0,9,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [30]:
columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet'
    )
""").df()

columns

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [31]:
con.sql(f"""
    SELECT
        MIN(report_date) AS oldest_date,
        MAX(report_date) AS newest_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,oldest_date,newest_date
0,2025-01-27,2026-06-30


In [33]:
for f in files:
  print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [38]:
from huggingface_hub import hf_hub_download

readme_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="README.md",
    repo_type="dataset",
    token=token
)

with open(readme_path, "r") as f:
    print(f.read())

---
license: other
language:
- en
tags:
- seo
- content-performance
- data-warehouse
- tabular
- education
- flyrank-internship
pretty_name: FlyRank Internship — Warehouse Star Schema (Pseudonymized, Gated)
size_categories:
- 10M<n<100M
extra_gated_prompt: >-
  By requesting access you agree to the FlyRank Internship Data Use Terms:
  anonymized research and education use only; no attempt to re-identify clients,
  domains, queries, keywords, or content; no redistribution of the raw data; and
  no client-identifying data in any public output (case study, repo, chart, or demo).
extra_gated_fields:
  Name: text
  Email: text
  Affiliation or cohort: text
  I agree to the FlyRank data-use terms: checkbox
configs:
- config_name: dim_clients
  data_files: dim_clients.parquet
- config_name: dim_content
  data_files: dim_content.parquet
- config_name: fact_content_daily_performance
  data_files: fact_content_daily_performance/**/*.parquet
- config_name: fact_content_query_90d
  data_files: fac

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

'''
Unit of analysis -> what exactly does one row mean?
One row represents the daily performance of one content for one client on a specific date.


Time window -> what dates are you analysing?
For this analysis, I will use data from 1 May 2026 to 31 May 2026. This month contains approximately 1.17 million records, making it the largest pre-June monthly dataset while keeping June 2026 untouched as the final test month.
'''

'\nUnit of analysis -> what exactly does one row mean?\nOne row represents the daily performance of one content for one client on a specific date. \n\n\nTime window -> what dates are you analysing?\nFor this analysis, I will use data from 1 May 2026 to 31 May 2026. This month contains approximately 1.17 million records, making it the largest pre-June monthly dataset while keeping June 2026 untouched as the final test month. \n'

In [13]:
con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        COUNT(*) AS row_count
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY 1
    ORDER BY row_count DESC
    LIMIT 20
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,row_count
0,2026-06-01,11694072
1,2026-05-01,11687376
2,2026-04-01,10424730
3,2026-03-01,9841378
4,2026-01-01,7890817
5,2025-12-01,7752930
6,2026-02-01,7355108
7,2025-11-01,6793825
8,2025-10-01,2165471
9,2025-09-01,845813


In [11]:
con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS row_count
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-05/**/*.parquet'
    )
""").df()

,start_date,end_date,row_count
0,2026-05-01,2026-05-31,11687376


In [6]:
from google.colab import userdata
token = userdata.get('HF_TOKEN')
print("Token loaded:", token is not None)

Token loaded: True


In [7]:
import duckdb
con = duckdb.connect()

# FlyRank warehouse root
rel = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [10]:
check_2 = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-05/**/*.parquet'
    )
    ORDER BY client_hash_id, content_hash_id, report_date
    LIMIT 20
""").df()

#client_hash_id -> smallest first, content_hash_id, then report date
check_2

,report_date,client_hash_id,content_hash_id
0,2026-05-25,client_04660893ae39614a,content_004de9653278b5a4
1,2026-05-26,client_04660893ae39614a,content_004de9653278b5a4
2,2026-05-27,client_04660893ae39614a,content_004de9653278b5a4
3,2026-05-28,client_04660893ae39614a,content_004de9653278b5a4
4,2026-05-29,client_04660893ae39614a,content_004de9653278b5a4
5,2026-05-30,client_04660893ae39614a,content_004de9653278b5a4
6,2026-05-31,client_04660893ae39614a,content_004de9653278b5a4
7,2026-05-22,client_04660893ae39614a,content_01410f2556c327ac
8,2026-05-23,client_04660893ae39614a,content_01410f2556c327ac
9,2026-05-24,client_04660893ae39614a,content_01410f2556c327ac


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
Label -> no field, since its something that I need to construct from future window, such as fuuture decline in impressions / clicks / position

Context -> report_date, client_hash_id, content_hash_id, gsc_data_available, ga4_data_available, client_has_gsc, client_has_ga4. This is the data that define valid rows and it should not be used as model signals.

Excluded -> gsc_sum_position, as its redundant (gsc_sum_position = gsc_impressions * gsc_avg_position)

Feature -> gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events
'''


In [15]:
con.sql(f"""
    SELECT
        sessions_organic,
        COUNT(*) AS row_count
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-05/**/*.parquet'
    )
    GROUP BY sessions_organic
    ORDER BY sessions_organic
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,sessions_organic,row_count
0,0,8934099
1,1,121543
2,2,100593
3,3,30429
4,4,27884
...,...,...
133,192,1
134,213,1
135,225,1
136,234,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''

'''


In [ ]:
#check grain

check_grain = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-05/**/*.parquet'
    )
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

check_grain

In [16]:
#check missing

check_missing = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS gsc_impressions_null,

        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS gsc_clicks_null,

        COUNT(*) FILTER (
            WHERE gsc_avg_position IS NULL
        ) AS gsc_avg_position_null,

        COUNT(*) FILTER (
            WHERE ga4_sessions IS NULL
        ) AS ga4_sessions_null,

        COUNT(*) FILTER (
            WHERE ga4_engaged_sessions IS NULL
        ) AS ga4_engaged_sessions_null

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-05/**/*.parquet'
    )
""").df()

check_missing

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_impressions_null,gsc_clicks_null,gsc_avg_position_null,ga4_sessions_null,ga4_engaged_sessions_null
0,11687376,0,0,7313954,2420302,2420302


In [17]:
#Check row counts and date window

check_window = con.sql(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS row_count
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-05/**/*.parquet'
    )
""").df()

check_window

,start_date,end_date,row_count
0,2026-05-01,2026-05-31,11687376


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

'''
History is unbalanced across clients because some clients have much longer GSC/GA4 histories than others. Some early rows contain only GSC data because GA4 was not yet available, so GA4 metrics can't be fairly compared across all rows. In addition, some feature windows may overlap with label window, which can create leakage if not handled carefully.
'''


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.